# Titanic Dataset

Most of the code cells below include comments explaining the task to be performed in those cells. Please delete the comments and add code to perform those tasks. There are a few code cells in which code has already been provided for you. In some cases, you will need to complete this code. 

# Import Statements

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import gc

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=RuntimeWarning)

# Load Training Data

In [ ]:
train = pd.read_csv('../input/titanic/train.csv')
train = train.sample(frac=1, random_state=1)

print(train.shape)

In [ ]:
train.head()

# Check for Missing Values

In [ ]:
train.isnull().sum().to_frame().T

# Check Label Distribution

In [ ]:
(train.Survived.value_counts() / len(train)).to_frame()

# Preprocessing

In [ ]:
train['FamSize'] = train['SibSp'] + train['Parch'] + 1

In [ ]:
train.head()

In [ ]:
def set_deck(cabin):
    if (str(cabin) == 'nan') or (str(cabin) == 'NaN'):
        return 'Missing'
    return str(cabin)

In [ ]:
train['Deck'] = train['Cabin'].map(set_deck)

In [ ]:
train.head()

In [ ]:
num_features = ['Age', 'FamSize', 'Fare']
cat_features = ['Sex', 'Pclass', 'Deck', 'Embarked']
features = num_features + cat_features

print(num_features)
print(cat_features)
print(features)

In [ ]:
num_transformer = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())  
    ]
)

cat_transformer = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

In [ ]:
y_train = train.Survived.values
train.drop(['Survived'], axis=1, inplace=True)

In [ ]:
preprocessor.fit(train[features])
X_train = preprocessor.transform(train[features])

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

# Model Selection

## Logistic Regression

In [ ]:
%%time 

lr_clf = LogisticRegression(max_iter=1000, solver='saga', penalty='elasticnet')

lr_parameters = {
    'l1_ratio':[0, 0.5, 1],
    'C': [.1, 1, 5, 20]
}

lr_grid = GridSearchCV(lr_clf, lr_parameters, cv=10, refit='True', n_jobs=-1, verbose=10, scoring='accuracy')
lr_grid.fit(X_train, y_train)

lr_model = lr_grid.best_estimator_

print('Best Parameters:', lr_grid.best_params_)
print('Best CV Score:  ', lr_grid.best_score_)
print('Training Acc:   ', lr_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

lr_summary = pd.DataFrame(lr_grid.cv_results_['params'])
lr_summary['cv_score'] = lr_grid.cv_results_['mean_test_score']

for r in lr_parameters['l1_ratio']:
    temp = lr_summary.query(f'l1_ratio == {r}')
    plt.plot(temp.C, temp.cv_score, label=r)
plt.xscale('log')
plt.ylim([0.75, 0.82])
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('CV Score')
plt.legend(title='L1 Ratio', loc='lower right')
plt.grid()
plt.show()

print(lr_summary.to_string(index=False))

       C  l1_ratio  cv_score
     0.1       0.0  0.807004
     0.1       0.5  0.804744
     0.1       1.0  0.800262
     1.0       0.0  0.809238
     1.0       0.5  0.805880
     1.0       1.0  0.808115
     5.0       0.0  0.811473
     5.0       0.5  0.812609
     5.0       1.0  0.810362
    20.0       0.0  0.811461
    20.0       0.5  0.810337
    20.0       1.0  0.808090

## Decision Trees

In [ ]:
%%time 

dt_clf = DecisionTreeClassifier(random_state=1)

dt_parameters = {
    'max_depth': [2, 6, 10, 14, 18, 24],
    'min_samples_leaf': [2, 4, 8, 16]
}

dt_grid = GridSearchCV(dt_clf, dt_parameters, cv=10, refit='True', n_jobs=-1, verbose=0, scoring='accuracy')
dt_grid.fit(X_train, y_train)

dt_model = dt_grid.best_estimator_

print('Best Parameters:', dt_grid.best_params_)
print('Best CV Score:  ', dt_grid.best_score_)
print('Training Acc:   ', dt_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

dt_summary = pd.DataFrame(dt_grid.cv_results_['params'])
dt_summary['cv_score'] = dt_grid.cv_results_['mean_test_score']

for ms in dt_parameters['min_samples_leaf']:
    temp = dt_summary.query(f'min_samples_leaf == {ms}')
    plt.plot(temp.max_depth, temp.cv_score, label=ms)
plt.xlabel('Maximum Depth')
plt.ylabel('CV Score')
plt.legend(title='Min Samples')
plt.grid()
plt.show()

print(dt_summary.to_string(index=False))

     max_depth  min_samples_leaf  cv_score
             2                 2  0.769963
             2                 4  0.769963
             2                 8  0.769963
             2                16  0.769963
             6                 2  0.807016
             6                 4  0.812622
             6                 8  0.813745
             6                16  0.793483
            10                 2  0.804707
            10                 4  0.799114
            10                 8  0.813745
            10                16  0.795730
            14                 2  0.793496
            14                 4  0.794644
            14                 8  0.809251
            14                16  0.795730
            18                 2  0.792372
            18                 4  0.798015
            18                 8  0.808127
            18                16  0.795730
            24                 2  0.793496
            24                 4  0.799139
            24                 8  0.808127
            24                16  0.795730

## Random Forests

In [ ]:
%%time 

rf_clf = RandomForestClassifier(random_state=1, n_estimators=40)

rf_parameters = {
    'max_depth': [4, 8, 16, 20, 24, 28, 32],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(rf_clf, rf_parameters, cv=10, refit='True', n_jobs=-1, verbose=0, scoring='accuracy')
rf_grid.fit(X_train, y_train)

rf_model = rf_grid.best_estimator_

print('Best Parameters:', rf_grid.best_params_)
print('Best CV Score:  ', rf_grid.best_score_)
print('Training Acc:   ', rf_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

rf_summary = pd.DataFrame(rf_grid.cv_results_['params'])
rf_summary['cv_score'] = rf_grid.cv_results_['mean_test_score']

for ms in rf_parameters['min_samples_leaf']:
    temp = rf_summary.query(f'min_samples_leaf == {ms}')
    plt.plot(temp.max_depth, temp.cv_score, label=ms)
plt.xlabel('Maximum Depth')
plt.ylabel('CV Score')
plt.legend(title='Min Samples')
plt.grid()
plt.show()

print(rf_summary.to_string(index=False))

     max_depth  min_samples_leaf  cv_score
             4                 1  0.801411
             4                 2  0.795805
             4                 4  0.734032
             8                 1  0.823870
             8                 2  0.813783
             8                 4  0.754232
            16                 1  0.828377
            16                 2  0.819401
            16                 4  0.760974
            20                 1  0.811573
            20                 2  0.817154
            20                 4  0.758727
            24                 1  0.812697
            24                 2  0.818277
            24                 4  0.758727
            28                 1  0.816030
            28                 2  0.818277
            28                 4  0.758727
            32                 1  0.810437
            32                 2  0.818277
            32                 4  0.758727

# Save Pipeline and Model

In [ ]:
print(rf_grid.best_params_)

In [ ]:
final_model = RandomForestClassifier(random_state=1, n_estimators=40, max_depth=16, min_samples_leaf=1)
final_model.fit(X_train, y_train)

print(final_model.score(X_train, y_train))

In [ ]:
joblib.dump(preprocessor, 'titanic_preprocessor_01.joblib')
joblib.dump(final_model, 'titanic_model_01.joblib')
print('Model written to file.')

In [ ]:
# Download both files to your local device and then upload them as a Kaggle dataset. 